# 🚀 CareerPilot – Model Training Notebook
End-to-end walkthrough: dataset → preprocessing → feature engineering → model training → evaluation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#1e293b'
matplotlib.rcParams['axes.facecolor'] = '#1e293b'
matplotlib.rcParams['text.color'] = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor'] = '#94a3b8'
matplotlib.rcParams['xtick.color'] = '#94a3b8'
matplotlib.rcParams['ytick.color'] = '#94a3b8'

print('✅ Imports OK')

## 1. Load / Generate Dataset

In [ ]:
from generate_dataset import generate_dataset

if os.path.exists('../data/dataset.csv'):
    df = pd.read_csv('../data/dataset.csv')
    print(f'Loaded existing dataset: {len(df)} rows')
else:
    df = generate_dataset(400)
    df.to_csv('../data/dataset.csv', index=False)
    print(f'Generated new dataset: {len(df)} rows')

df.head()

## 2. Exploratory Data Analysis

In [ ]:
print(df.describe())
print('\nRole distribution:')
print(df['role'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ATS score distribution
axes[0].hist(df['ats_score'], bins=25, color='#3b82f6', edgecolor='#1e293b')
axes[0].set_title('ATS Score Distribution', color='#e2e8f0')
axes[0].set_xlabel('ATS Score')
axes[0].set_ylabel('Count')

# ATS score by role
role_means = df.groupby('role')['ats_score'].mean().sort_values()
axes[1].barh(role_means.index, role_means.values, color='#22c55e')
axes[1].set_title('Avg ATS Score by Role', color='#e2e8f0')
axes[1].set_xlabel('Mean ATS Score')

plt.tight_layout()
plt.show()

## 3. Text Preprocessing

In [ ]:
from src.preprocessing import clean_text

sample = df['resume_text'].iloc[0]
print('ORIGINAL (first 300 chars):')
print(sample[:300])
print('\nCLEANED:')
print(clean_text(sample)[:300])

In [ ]:
df['cleaned_text'] = df['resume_text'].apply(lambda t: clean_text(t, remove_stops=True))
df['text_length'] = df['cleaned_text'].apply(len)
df['word_count'] = df['cleaned_text'].apply(lambda t: len(t.split()))

print(df[['text_length','word_count','ats_score']].describe())

## 4. Feature Engineering – TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

X = vectorizer.fit_transform(df['cleaned_text'])
y = df['ats_score'].values

print(f'Feature matrix shape: {X.shape}')
print(f'Top 20 features: {vectorizer.get_feature_names_out()[:20]}')

## 5. Model Training

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')

# Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = np.clip(rf.predict(X_test), 0, 100)

# Ridge baseline
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = np.clip(ridge.predict(X_test), 0, 100)

print('\n── Random Forest ──')
print(f'MAE : {mean_absolute_error(y_test, rf_pred):.2f}')
print(f'R²  : {r2_score(y_test, rf_pred):.3f}')

print('\n── Ridge Regression ──')
print(f'MAE : {mean_absolute_error(y_test, ridge_pred):.2f}')
print(f'R²  : {r2_score(y_test, ridge_pred):.3f}')

## 6. Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Actual vs Predicted
axes[0].scatter(y_test, rf_pred, alpha=0.5, color='#3b82f6', s=30)
axes[0].plot([20,100],[20,100], 'r--', lw=1)
axes[0].set_xlabel('Actual ATS Score')
axes[0].set_ylabel('Predicted ATS Score')
axes[0].set_title('Actual vs Predicted (Random Forest)', color='#e2e8f0')

# Residuals
residuals = y_test - rf_pred
axes[1].hist(residuals, bins=25, color='#f59e0b', edgecolor='#1e293b')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', color='#e2e8f0')

plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump(rf, '../models/ats_model.pkl')
joblib.dump(vectorizer, '../models/vectorizer.pkl')
print('✅ Model and vectorizer saved to /models')

## 8. Quick Inference Test

In [ ]:
from src.ats_score import predict_ats_score
from src.job_matcher import compute_match_score
from src.skill_extractor import extract_skills

test_resume = """
Priya Verma | priya.verma@gmail.com | +91 9988776655
B.Tech Computer Science | 2021
Skills: Python, Machine Learning, TensorFlow, SQL, Pandas, Docker, Git
Experience: Data Scientist at Infosys (2021-2024)
"""
test_jd = "Hiring a Data Scientist skilled in Python, ML, NLP, TensorFlow, SQL, Docker, AWS."

print(f"ATS Score   : {predict_ats_score(test_resume):.1f}")
print(f"Match Score : {compute_match_score(test_resume, test_jd):.1f}%")
print(f"Skills      : {extract_skills(test_resume)}")